In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit, current_timestamp,col
from pyspark.sql.utils import AnalysisException

### Función auxiliar para registrar metadatos y logs

In [0]:
%run "./00_utils_logging_functions"

In [0]:
path_students   = "/Workspace/proyectos/analisis_estudiantes/data_raw/students.csv"
path_enrollments = "/Workspace/proyectos/analisis_estudiantes/data_raw/enrollments.csv"
path_subjects    = "/Workspace/proyectos/analisis_estudiantes/data_raw/subjects.csv"

job_name = "control_ingesta"
layer = "Landing"



### Funciones utiles para el proyecto

Creacion de la tabla **landing_raw_students** donde se le añade dos nuevas columnas _id_ingestion_ y _timestamp_ingestion_ con el objetivo de supervisar los datos que se van introduciendo al proyecto.

In [0]:

section = "_students"


table_name = "workspace.dbestudiantes.landing_raw_students"

try:

    df_raw_students = spark.read.csv(path_students, header=True, inferSchema=True)

    w = Window.orderBy(lit(1))

    df_landing_raw_students = (
        df_raw_students
        .withColumn("id_ingestion", row_number().over(w))
        .withColumn("timestamp_ingestion", current_timestamp())
    )

    if not spark.catalog.tableExists(table_name):
        
        

        df_landing_raw_students.write.format("delta").mode("overwrite").saveAsTable("workspace.dbestudiantes.landing_raw_students")

        id_ingesta_last = df_landing_raw_students.select("id_ingestion").orderBy(col("id_ingestion").desc()).limit(1).collect()[0][0]

        row = df_landing_raw_students.count()
      

        log_meta(job_name+section , layer, row, id_ingesta_last, "SUCCESS")
        log_event(job_name+section, layer, "INFO", "Tabla landing_raw_students creada.")

       


       
    
    else:
     
        #Leer el ultimo id_ingestion de la tabla landing_raw_students
        df_landing_raw_students_last = (
            spark.read.table(table_name)
            .select("*")
            .orderBy(col("id_ingestion").desc())
            .limit(1)
            
        )
        if df_landing_raw_students_last.count() == 0:
            
            log_event(job_name+section, layer, "INFO", "No se puede obtener la última ingesta porque no hay ninguna.")
        else:
            df_landing_raw_students_last = df_landing_raw_students_last.limit(1)
            display(df_landing_raw_students_last)
            id_ingesta_last = df_landing_raw_students_last.select("id_ingestion").collect()[0][0]

            df_landing_raw_students_nuevos = df_landing_raw_students.filter(col("id_ingestion") > id_ingesta_last)
            row = df_landing_raw_students_nuevos.count()
   
            id_ingesta_last = id_ingesta_last + row
            
            
            log_meta(job_name+section , layer, row, id_ingesta_last, "SUCCESS")

            df_landing_raw_students_nuevos.write.format("delta").mode("append").saveAsTable("workspace.dbestudiantes.landing_raw_students")
            log_event(job_name+section, layer, "INFO", "Tabla landing_raw_students actualizada.")
        
       
            
    
except Exception as e:
    log_meta(job_name+section , layer, -1, -1, "ERROR")
    log_event(job_name+section, layer, "ERROR", f"Se ha producido un Error {e}")
    raise

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Creacion de la tabla **landing_raw_enrollments** donde se le añade dos nuevas columnas _id_ingestion_ y _timestamp_ingestion_ con el objetivo de supervisar los datos que se van introduciendo al proyecto.

In [0]:
section = "_enrollments"


table_name = "workspace.dbestudiantes.landing_raw_enrollments"

try:

    df_raw_enrollments = spark.read.csv(path_enrollments, header=True, inferSchema=True)

    w = Window.orderBy(lit(1))

    df_landing_raw_enrollments = (
        df_raw_enrollments
        .withColumn("id_ingestion", row_number().over(w))
        .withColumn("timestamp_ingestion", current_timestamp())
    )

    if not spark.catalog.tableExists(table_name):
      
        

        df_landing_raw_enrollments.write.format("delta").mode("overwrite").saveAsTable("workspace.dbestudiantes.landing_raw_enrollments")

        id_ingesta_last = df_landing_raw_enrollments.select("id_ingestion").orderBy(col("id_ingestion").desc()).limit(1).collect()[0][0]

        row = df_landing_raw_enrollments.count()
      

        log_meta(job_name+section , layer, row, id_ingesta_last, "SUCCESS")
        log_event(job_name+section, layer, "INFO", "Tabla landing_raw_enrollments creada.")

    
    else:
        
        df_landing_raw_enrollments_last = (
            spark.read.table(table_name)
            .select("*")
            .orderBy(col("id_ingestion").desc())
            .limit(1)
            
        )
        if df_landing_raw_enrollments_last.count() == 0:
            
            log_event(job_name+section, layer, "INFO", "No se puede obtener la última ingesta porque no hay ninguna.")
        else:
            df_landing_raw_enrollments_last = df_landing_raw_enrollments_last.limit(1)
      
            id_ingesta_last = df_landing_raw_enrollments_last.select("id_ingestion").collect()[0][0]

            df_landing_raw_enrollments_nuevos = df_landing_raw_enrollments.filter(col("id_ingestion") > id_ingesta_last)
            row = df_landing_raw_enrollments_nuevos.count()
    
            id_ingesta_last = id_ingesta_last + row
          
            
            log_meta(job_name+section , layer, row, id_ingesta_last, "SUCCESS")

            df_landing_raw_enrollments_nuevos.write.format("delta").mode("append").saveAsTable("workspace.dbestudiantes.landing_raw_enrollments")
            log_event(job_name+section, layer, "INFO", "Tabla landing_raw_enrollments actualizada.")
       

    
except Exception as e:
    log_meta(job_name+section, layer, -1, -1, "ERROR")
    log_event(job_name+section, layer, "ERROR", f"Se ha producido un Error {e}")
    raise

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Creacion de la tabla **landing_raw_subjects** donde se le añade dos nuevas columnas _id_ingestion_ y _timestamp_ingestion_ con el objetivo de supervisar los datos que se van introduciendo al proyecto.

In [0]:
section = "_subjects"


table_name = "workspace.dbestudiantes.landing_raw_subjects"

try:

    df_raw_subjects = spark.read.csv(path_subjects, header=True, inferSchema=True)

    w = Window.orderBy(lit(1))

    df_landing_raw_subjects = (
        df_raw_subjects
        .withColumn("id_ingestion", row_number().over(w))
        .withColumn("timestamp_ingestion", current_timestamp())
    )

    if not spark.catalog.tableExists(table_name):
      
        

        df_landing_raw_subjects.write.format("delta").mode("overwrite").saveAsTable("workspace.dbestudiantes.landing_raw_subjects")

        id_ingesta_last = df_landing_raw_subjects.select("id_ingestion").orderBy(col("id_ingestion").desc()).limit(1).collect()[0][0]

        row = df_landing_raw_subjects.count()
      

        log_meta(job_name+section , layer, row, id_ingesta_last, "SUCCESS")
        log_event(job_name+section, layer, "INFO", "Tabla landing_raw_subjects creada.")

    
    else:
        
        df_landing_raw_subjects_last = (
            spark.read.table(table_name)
            .select("*")
            .orderBy(col("id_ingestion").desc())
            .limit(1)
            
        )
        if df_landing_raw_subjects_last.count() == 0:
            
            log_event(job_name+section, layer, "INFO", "No se puede obtener la última ingesta porque no hay ninguna.")
        else:
            df_landing_raw_subjects_last = df_landing_raw_subjects_last.limit(1)
      
            id_ingesta_last = df_landing_raw_subjects_last.select("id_ingestion").collect()[0][0]

            df_landing_raw_subjects_nuevos = df_landing_raw_subjects.filter(col("id_ingestion") > id_ingesta_last)
            row = df_landing_raw_subjects_nuevos.count()
    
            id_ingesta_last = id_ingesta_last + row
          
            
            log_meta(job_name+section, layer, row, id_ingesta_last, "SUCCESS")

            df_landing_raw_subjects_nuevos.write.format("delta").mode("append").saveAsTable("workspace.dbestudiantes.landing_raw_subjects")
            log_event(job_name+section, layer, "INFO", "Tabla landing_raw_subjects actualizada.")
       

    
except Exception as e:
    log_meta(job_name+section, layer, -1, -1, "ERROR")
    log_event(job_name+section, layer, "ERROR", f"Se ha producido un Error {e}")
    raise

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
